Copyright Amazon.com, Inc. or its affiliates. All Rights Reserved.
SPDX-License-Identifier: MIT-0

# Module 5: Observe & Evaluate with CloudWatch Omni

Modules 2 to 4 gave each platform layer its own observability surface: LiteLLM spend logs, interceptor
audit logs, the GenAI Observability dashboard, Memory events. None of them answers the two questions a
platform team is asked first: *what did this agent do, end to end, for this user?* and *is it getting
better or worse?*

Amazon CloudWatch Omni answers both. It reads the telemetry the platform already emits (no
re-instrumentation), correlates logs, metrics and traces in one CloudWatch Dataset, and scores agent
runs with evaluators served by Amazon Bedrock AgentCore Evaluations.

| Concept | What it is | In this workshop |
|---------|-----------|------------------|
| Domain | Identity and sign-in boundary, one per account, `https://<name>.cloudwatch-omni.global.app.aws` | One account domain, IAM sign-in only |
| Space | Where you work, one per account and Region | `workshop-space` |
| CloudWatch Dataset | Correlated, queryable logs, metrics and traces | Every log group in the account plus `aws/spans` |

The workshop account is a single account, so this module uses the single-account setup. The
[enterprise blueprint](https://github.com/aws-samples/sample-ai-agent-factory/tree/main/enterprise-agentic-ai-platform-blueprint)
implements the multi-account hub model (organization domain, centralization rules into an observability account).

## Step 1: Verify the kernel

In [ ]:
%pip install --quiet "boto3==1.43.126" "botocore==1.43.126"
import boto3

assert tuple(int(x) for x in boto3.__version__.split(".")[:3]) >= (1, 43, 100), (
    f"boto3 too old: {boto3.__version__} (need >=1.43.100 for the cloudwatchomni client)"
)
# If the kernel imported an older botocore before the pip install above, the new
# service models are not loaded yet. Fail here with an actionable message.
services = boto3.session.Session().get_available_services()
assert "cloudwatchomni" in services and "observabilityadmin" in services, (
    "This kernel has no cloudwatchomni client. Restart the kernel (Kernel > Restart) and run this cell again."
)
print(f"boto3 {boto3.__version__} OK - cloudwatchomni and observabilityadmin clients available")

## Step 2: Environment setup

In [ ]:
import json
import os
import pathlib
import time

import boto3

REGION = (
    os.environ.get("AWS_REGION")
    or os.environ.get("AWS_DEFAULT_REGION")
    or boto3.session.Session().region_name
)
if not REGION:
    raise RuntimeError("No AWS region configured. Set AWS_REGION or run `aws configure set region <region>`.")
ACCOUNT_ID = boto3.client("sts").get_caller_identity()["Account"]

# The CLI pages save identifiers as `export KEY=VALUE` lines in this file, so a
# participant can switch between the CLI and notebook paths at any step.
STATE_FILE = pathlib.Path.home() / ".workshop-omni.env"


def load_state():
    state = {}
    if STATE_FILE.exists():
        for line in STATE_FILE.read_text().splitlines():
            if line.startswith("export ") and "=" in line:
                key, value = line[len("export "):].split("=", 1)
                state[key] = value.strip()
    return state


def save_state(**values):
    state = load_state()
    state.update({k: v for k, v in values.items() if v})
    STATE_FILE.write_text("".join(f"export {k}={v}\n" for k, v in state.items()))
    return state


omni = boto3.client("cloudwatchomni", region_name=REGION)
obs_admin = boto3.client("observabilityadmin", region_name=REGION)
logs = boto3.client("logs", region_name=REGION)
cfn = boto3.client("cloudformation", region_name=REGION)
print(f"Region {REGION}, account ...{ACCOUNT_ID[-4:]}")

## Step 3: What is already in place

Workshop Studio provisioned three things for this module in `workshop-agentcore-stack`:

- CloudWatch Transaction Search at 100% indexing, the one-time prerequisite for any AgentCore span to
  reach CloudWatch (it takes about 10 minutes to apply, so it was done during provisioning)
- `CloudWatchOmniSpaceAccessRole-workshop` and `CloudWatchOmniDatasetIntegrationRole-workshop`, the
  roles Omni and CloudWatch Logs assume for your space
- `AgentCoreEvaluationRole-workshop`, the execution role AgentCore Evaluations uses

Confirm all three.

In [ ]:
xray = boto3.client("xray", region_name=REGION)
destination = xray.get_trace_segment_destination()
print(f"Trace destination: {destination['Destination']} ({destination['Status']})")
assert destination["Destination"] == "CloudWatchLogs", (
    "Transaction Search is not on. Deploy static/cfn/agentcore/workshop-agentcore-stack.yaml first; "
    "participants do not have permission to change this setting by hand."
)

outputs = {o["OutputKey"]: o["OutputValue"]
           for o in cfn.describe_stacks(StackName="workshop-agentcore-stack")["Stacks"][0]["Outputs"]}
for key in ("OmniSpaceAccessRoleArn", "OmniDatasetIntegrationRoleArn", "AgentCoreEvaluationRoleArn"):
    assert key in outputs, f"{key} missing: the stack predates Module 5"
    print(f"{key:32s} {outputs[key]}")

## Tracks

- **Track 1** (agent only): notebooks 02, 03, then 05 to 08.
- **Track 2** (platform only): notebooks 02, 03, 04, 08.
- **Track 3**: all of them.

Continue with `02-create-space.ipynb`.